In [57]:
import pandas as pd
import numpy as np
import os

In [58]:
oteot = pd.read_excel("Data\CategoryWiseOverTime.xlsx", sheet_name="Report_CategoryWiseOTWoven")

In [59]:
# =============================================================================
# Category Wise Over Time Report
# Clean Report_CategoryWiseOTWoven -> return `oteot` DataFrame only
# =============================================================================

import re
import warnings
from datetime import date, datetime
from pathlib import Path

import numpy as np
import pandas as pd


# -----------------------------------------------------------------------------
# Constants
# -----------------------------------------------------------------------------

_MONTHS = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]

# Excel 1900 date system
_EXCEL_EPOCH = pd.Timestamp("1899-12-30")

# Only whole-number values inside this range are treated as Excel dates
# 2000-01-01 -> 2100-12-31
_MIN_SERIAL = 36526
_MAX_SERIAL = 73415

# Expected final date format
_DATE_RE = re.compile(r"^\d{2}-[A-Z][a-z]{2}-\d{4}$")

# Header matching keys
_HOUR_KEYS = {"hr", "hour", "hours"}
_AMNT_KEYS = {"amnt", "amount"}


# -----------------------------------------------------------------------------
# Helper Functions
# -----------------------------------------------------------------------------

def _clean_cell(value):
    """
    Normalize one Excel cell.

    Rules:
    - Remove zero-width characters
    - Remove line breaks/tabs
    - Collapse multiple spaces
    - Trim leading/trailing spaces
    - Convert blank strings / None / NaN to np.nan
    - Keep numbers and dates unchanged
    """

    if isinstance(value, str):
        value = re.sub(r"[\u200b\ufeff]", "", value)
        value = re.sub(r"\s+", " ", value).strip()

        return np.nan if value == "" else value

    if value is None or pd.isna(value):
        return np.nan

    return value


def _key(value):
    """
    Convert header text into a comparison key.

    Examples:
        'Hr.'          -> 'hr'
        'Hour'         -> 'hour'
        'Amnt.'        -> 'amnt'
        'Month Total'  -> 'monthtotal'
        'Factory'      -> 'factory'
    """

    return re.sub(r"[^a-z]", "", str(value).lower())


def _fmt_date(value):
    """
    Convert a date-like value into dd-MMM-yyyy format.
    """

    return (
        f"{value.day:02d}-"
        f"{_MONTHS[value.month - 1]}-"
        f"{value.year}"
    )


def _to_date_label(value):
    """
    Convert:
        - datetime/date/Timestamp
        - Excel serial date

    into:
        dd-MMM-yyyy

    Return None if the value is not a valid date.
    """

    # Real Python/Pandas date
    if isinstance(value, (datetime, date, pd.Timestamp)):
        return _fmt_date(value)

    # Excel serial number
    if (
        isinstance(value, (int, float, np.integer, np.floating))
        and not isinstance(value, bool)
    ):
        numeric_value = float(value)

        if (
            numeric_value.is_integer()
            and _MIN_SERIAL <= numeric_value <= _MAX_SERIAL
        ):
            converted_date = (
                _EXCEL_EPOCH
                + pd.Timedelta(days=int(numeric_value))
            )

            return _fmt_date(converted_date)

    return None


def _ffill(values):
    """
    Forward-fill values without relying on pandas downcasting.
    """

    output = []
    last_value = np.nan

    for value in values:

        if pd.isna(value):
            output.append(last_value)

        else:
            last_value = value
            output.append(value)

    return output


# -----------------------------------------------------------------------------
# Main Function
# -----------------------------------------------------------------------------

def clean_category_wise_overtime(
    input_file,
    source_sheet="Report_CategoryWiseOTWoven",
    rows_to_drop=11,
    fill_columns=("Factory", "Type", "Category"),
):
    """
    Read and clean the Category Wise Over Time Excel report.

    Returns
    -------
    pandas.DataFrame
        Cleaned `oteot` DataFrame.

    Final structure
    ---------------

    Row 0:
        Month Total | ... | 23-Sep-2026 | ... | 24-Sep-2026 | ...

    Row 1:
        Factory | Type | Category | Type | Emp | Hour | Amnt. | Emp | Hr. | Amnt. | ...

    Row 2+:
        Actual OT/EOT data

    No Excel file is created or saved.
    """

    # =========================================================================
    # 1. Validate input file
    # =========================================================================

    input_file = Path(input_file)

    if not input_file.exists():
        raise FileNotFoundError(
            f"Input file not found: {input_file}"
        )

    # =========================================================================
    # 2. Read source worksheet
    # =========================================================================

    with pd.ExcelFile(input_file) as excel:

        if source_sheet not in excel.sheet_names:
            raise ValueError(
                f"Sheet '{source_sheet}' not found.\n"
                f"Available sheets: {excel.sheet_names}"
            )

        raw = excel.parse(
            source_sheet,
            header=None,
            dtype=object,
            keep_default_na=False
        )

    # =========================================================================
    # 3. Remove report information rows
    #
    # Example:
    # rows_to_drop = 11
    #
    # iloc[11:] means:
    # Excel row 12 onward
    # =========================================================================

    df = (
        raw
        .iloc[rows_to_drop:]
        .reset_index(drop=True)
    )

    if df.empty:
        raise ValueError(
            f"The sheet has no rows after removing "
            f"the first {rows_to_drop} rows."
        )

    # Use zero-based integer column indexes
    df.columns = range(df.shape[1])

    # =========================================================================
    # 4. Normalize every cell
    # =========================================================================

    elementwise = (
        df.map
        if hasattr(df, "map")
        else df.applymap
    )

    df = elementwise(_clean_cell).astype(object)

    # =========================================================================
    # 5. Remove completely blank rows
    # =========================================================================

    df = (
        df
        .dropna(how="all")
        .reset_index(drop=True)
    )

    if df.empty:
        raise ValueError(
            "The worksheet became empty after removing blank rows."
        )

    # =========================================================================
    # 6. Convert Excel serial dates in first row
    #
    # Example:
    # 46288 -> 23-Sep-2026
    # =========================================================================

    for column in df.columns:

        date_label = _to_date_label(
            df.iat[0, column]
        )

        if date_label:
            df.iat[0, column] = date_label

    # =========================================================================
    # 7. Locate actual header row
    #
    # Expected:
    #
    # Row 0 -> Month Total / Dates
    # Row 1 -> Factory / Type / Category / Emp / Hr. / Amnt.
    # =========================================================================

    header_rows = [
        row_index
        for row_index in df.index[:5]
        if any(
            _key(value) == "factory"
            for value in df.loc[row_index].dropna()
        )
    ]

    if header_rows != [1]:

        raise ValueError(
            "Expected the date row at position 0 and the "
            "'Factory ... Emp/Hr./Amnt.' row at position 1 "
            f"after removing {rows_to_drop} rows.\n"
            f"Found header row(s): {header_rows}.\n"
            "Please check `rows_to_drop`."
        )

    # =========================================================================
    # 8. Create header lookup
    # =========================================================================

    hdr = {
        column: _key(value)
        for column, value in df.loc[1].items()
        if pd.notna(value)
    }

    # =========================================================================
    # 9. Find all Emp columns
    #
    # Every OT/EOT group is:
    #
    # Emp | Hr. | Amnt.
    #
    # Example:
    #
    # Emp | Hr. | Amnt. | Emp | Hr. | Amnt.
    # =========================================================================

    emp_cols = sorted(
        column
        for column, header_key in hdr.items()
        if header_key == "emp"
    )

    if not emp_cols:
        raise ValueError(
            "No 'Emp' columns found in the header row."
        )

    # =========================================================================
    # 10. Detect every Emp/Hr./Amnt. group
    # =========================================================================

    bounds = emp_cols[1:] + [df.shape[1]]

    groups = []

    for emp_column, next_column in zip(
        emp_cols,
        bounds
    ):

        # Find Hour/Hr. column
        hr_column = next(
            (
                column
                for column in range(
                    emp_column + 1,
                    next_column
                )
                if hdr.get(column) in _HOUR_KEYS
            ),
            None
        )

        # Find Amnt./Amount column
        amount_column = next(
            (
                column
                for column in range(
                    emp_column + 1,
                    next_column
                )
                if (
                    hdr.get(column) in _AMNT_KEYS
                    and hr_column is not None
                    and column > hr_column
                )
            ),
            None
        )

        if hr_column is None or amount_column is None:

            raise ValueError(
                "Column group starting at column index "
                f"{emp_column} does not contain a valid "
                "Emp / Hr. / Amnt. structure."
            )

        groups.append(
            {
                "emp": emp_column,
                "hr": hr_column,
                "amnt": amount_column,
                "label": None,
                "kind": None,
            }
        )

    # =========================================================================
    # 11. Read Month Total / Date labels from first row
    # =========================================================================

    labels = []

    for column, value in df.loc[0].items():

        if pd.isna(value):
            continue

        value_key = _key(value)

        # Month Total
        if value_key == "monthtotal":

            labels.append(
                (
                    column,
                    "Month Total",
                    "month_total"
                )
            )

        # Date
        elif (
            isinstance(value, str)
            and _DATE_RE.match(value)
        ):

            labels.append(
                (
                    column,
                    value,
                    "date"
                )
            )

    # =========================================================================
    # 12. Validate date headers
    # =========================================================================

    if not any(
        kind == "date"
        for _, _, kind in labels
    ):

        raise ValueError(
            "No date headers found in the first row."
        )

    if not any(
        kind == "month_total"
        for _, _, kind in labels
    ):

        warnings.warn(
            "'Month Total' was not found in the first row."
        )

    # =========================================================================
    # 13. Attach Month Total / Date labels to Emp groups
    # =========================================================================

    for column, text, kind in labels:

        # Find first Emp group whose Emp column is at or
        # to the right of the label
        group = next(
            (
                group
                for group in groups
                if group["emp"] >= column
            ),
            None
        )

        if group is None:

            raise ValueError(
                f"Header '{text}' at column index "
                f"{column} cannot be matched to an "
                "Emp/Hr./Amnt. group."
            )

        if group["label"] is not None:

            raise ValueError(
                f"Header '{text}' is being assigned to an "
                "Emp/Hr./Amnt. group that already has "
                f"the label '{group['label']}'."
            )

        # Ensure Hr. and Amnt. cells are blank
        if any(
            pd.notna(df.iat[0, column_index])
            for column_index in (
                group["hr"],
                group["amnt"]
            )
        ):

            raise ValueError(
                f"Unexpected content found after "
                f"'{text}'. Refusing to overwrite data."
            )

        group["label"] = text
        group["kind"] = kind

        # Move label to Emp column if necessary
        if column != group["emp"]:

            df.iat[0, column] = np.nan
            df.iat[0, group["emp"]] = text

    # =========================================================================
    # 14. Check groups without labels
    # =========================================================================

    for group in groups:

        if group["label"] is None:

            warnings.warn(
                "Emp/Hr./Amnt. group at column index "
                f"{group['emp']} has no date header."
            )

    # =========================================================================
    # 15. Forward-fill Factory / Type / Category
    #
    # IMPORTANT:
    # Only the FIRST matching header is selected.
    #
    # This prevents the second 'Type' column, which represents
    # OT/EOT type, from being incorrectly forward-filled.
    # =========================================================================

    fill_cols = []

    for name in fill_columns:

        target_key = _key(name)

        column = next(
            (
                column
                for column, header_key in sorted(hdr.items())
                if header_key == target_key
            ),
            None
        )

        if column is None:

            raise ValueError(
                f"Could not find '{name}' "
                "header for forward-fill."
            )

        fill_cols.append(column)

    # Only actual data rows: row 2 onward
    for column in fill_cols:

        df.loc[2:, column] = _ffill(
            df.loc[2:, column].tolist()
        )

    # =========================================================================
    # 16. Remove completely blank columns
    #
    # These are normally leftovers from merged Excel cells.
    # =========================================================================

    keep_columns = [
        column
        for column in df.columns
        if df[column].notna().any()
    ]

    if not keep_columns:

        raise ValueError(
            "No non-blank columns remain after cleaning."
        )

    # Map original column positions -> new positions
    new_position = {
        old_column: new_column
        for new_column, old_column
        in enumerate(keep_columns)
    }

    # =========================================================================
    # 17. Create final `oteot` DataFrame
    # =========================================================================

    oteot = df[keep_columns].copy()

    # Reset columns to zero-based integers
    oteot.columns = range(
        len(keep_columns)
    )

    # =========================================================================
    # 18. Update group positions after blank-column removal
    # =========================================================================

    for group in groups:

        try:
            group["emp"] = new_position[
                group["emp"]
            ]

            group["hr"] = new_position[
                group["hr"]
            ]

            group["amnt"] = new_position[
                group["amnt"]
            ]

        except KeyError as exc:

            raise ValueError(
                "A required Emp/Hr./Amnt. column "
                "was removed unexpectedly during "
                "blank-column cleanup."
            ) from exc

        # Every group must remain:
        #
        # Emp | Hr. | Amnt.
        #
        if not (
            group["hr"] == group["emp"] + 1
            and
            group["amnt"] == group["emp"] + 2
        ):

            raise ValueError(
                f"Group '{group['label']}' is not arranged "
                "as three adjacent columns after cleaning."
            )

    # =========================================================================
    # 19. Final DataFrame cleanup
    # =========================================================================

    oteot = oteot.reset_index(drop=True)

    # =========================================================================
    # 20. Return ONLY the DataFrame
    # =========================================================================

    return oteot


# =============================================================================
# Run
# =============================================================================

oteot = clean_category_wise_overtime(
    input_file=r"Data\CategoryWiseOverTime.xlsx",
    source_sheet="Report_CategoryWiseOTWoven",
    rows_to_drop=11,
)

display(oteot.head(20))

,0,1,2,3,4,5,6,7,8,9,...,66,67,68,69,70,71,72,73,74,75
0,NaN,NaN,NaN,NaN,Month Total,NaN,NaN,23-Sep-2026,NaN,NaN,...,NaN,03-Sep-2026,NaN,NaN,02-Sep-2026,NaN,NaN,01-Sep-2026,NaN,NaN
1,Factory,Type,Category,Type,Emp,Hour,Amnt.,Emp,Hr.,Amnt.,...,Amnt.,Emp,Hr.,Amnt.,Emp,Hr.,Amnt.,Emp,Hr.,Amnt.
2,MGL,Knit,Cutting,OT,604,1187.5,95210.77,47,88.5,7084.25,...,NaN,38,76,6082.89,45,89,7107.37,28,54,4371.6
3,MGL,Knit,Cutting,EOT,268,453.5,36771.43,0,0,0,...,NaN,36,35.5,2849.09,44,52,4201.1,14,14,1168.6
4,MGL,Knit,Finishing,OT,2173,4333.5,349417.42,21,42,3477.6,...,16272.2,112,223,17911.66,109,218,17502.55,106,210,16897.88
5,MGL,Knit,Finishing,EOT,1620,5037.5,407402.87,15,51,4226.89,...,47741.11,111,313,25273.69,109,317,25611.35,104,302,24346.16
6,MGL,Knit,Line 11 Helper,OT,121,239,17000.59,2,4,269.29,...,NaN,12,24,1689.54,10,20,1408.64,6,11,805.11
7,MGL,Knit,Line 11 Helper,EOT,58,122,8693.5,0,0,0,...,NaN,12,20,1396,9,17,1173.72,5,5,364.28
8,MGL,Knit,Line 11 Iron,OT,26,51,3890.62,1,2,153.09,...,NaN,2,4,322.17,3,6,462.62,2,3,237.63
9,MGL,Knit,Line 11 Iron,EOT,13,28,2104.03,0,0,0,...,NaN,1,3,229.63,3,6,448.3,1,1,76.54


In [60]:
display(oteot.head(2))

,0,1,2,3,4,5,6,7,8,9,...,66,67,68,69,70,71,72,73,74,75
0,NaN,NaN,NaN,NaN,Month Total,NaN,NaN,23-Sep-2026,NaN,NaN,...,NaN,03-Sep-2026,NaN,NaN,02-Sep-2026,NaN,NaN,01-Sep-2026,NaN,NaN
1,Factory,Type,Category,Type,Emp,Hour,Amnt.,Emp,Hr.,Amnt.,...,Amnt.,Emp,Hr.,Amnt.,Emp,Hr.,Amnt.,Emp,Hr.,Amnt.


In [61]:
# Syntax: df.iat[row_position, column_position] = new_value
oteot.iat[1, 1] = "Factory_Type" 
oteot.iat[1, 3] = "OT/EOT"


In [62]:
display(oteot.head(5))

,0,1,2,3,4,5,6,7,8,9,...,66,67,68,69,70,71,72,73,74,75
0,NaN,NaN,NaN,NaN,Month Total,NaN,NaN,23-Sep-2026,NaN,NaN,...,NaN,03-Sep-2026,NaN,NaN,02-Sep-2026,NaN,NaN,01-Sep-2026,NaN,NaN
1,Factory,Factory_Type,Category,OT/EOT,Emp,Hour,Amnt.,Emp,Hr.,Amnt.,...,Amnt.,Emp,Hr.,Amnt.,Emp,Hr.,Amnt.,Emp,Hr.,Amnt.
2,MGL,Knit,Cutting,OT,604,1187.5,95210.77,47,88.5,7084.25,...,NaN,38,76,6082.89,45,89,7107.37,28,54,4371.6
3,MGL,Knit,Cutting,EOT,268,453.5,36771.43,0,0,0,...,NaN,36,35.5,2849.09,44,52,4201.1,14,14,1168.6
4,MGL,Knit,Finishing,OT,2173,4333.5,349417.42,21,42,3477.6,...,16272.2,112,223,17911.66,109,218,17502.55,106,210,16897.88


In [63]:
# =============================================================================
# oteot -> OT / Extra-OT Management Report
# Returns pandas DataFrame only
# No Excel export / No file download
# =============================================================================

import re
import numpy as np
import pandas as pd


# =============================================================================
# Fixed zero-based column positions inside `oteot`
#
# Row 0 = Date / reporting-period information
# Row 1 = Actual column headers
# Row 2 onward = Data
# =============================================================================

COLS = {
    "Factory": 0,
    "Type": 1,
    "Category": 2,
    "OTEOT": 3,

    # Month Total
    "MT_Emp": 4,
    "MT_Hour": 5,
    "MT_Amnt": 6,

    # Previous Day
    "PD_Emp": 7,
    "PD_Hour": 8,
    "PD_Amnt": 9,
}


NUMERIC_COLS = [
    "MT_Emp",
    "MT_Hour",
    "MT_Amnt",
    "PD_Emp",
    "PD_Hour",
    "PD_Amnt",
]


# =============================================================================
# KPI definitions
# =============================================================================

KPI_SPEC = [
    (
        "Number of OT Employees",
        "PD_Emp",
        "OT",
    ),

    (
        "Number of Extra OT Employees",
        "PD_Emp",
        "EOT",
    ),

    (
        "Previous Days' OT Hours",
        "PD_Hour",
        "OT",
    ),

    (
        "Previous Days' Extra OT Hours",
        "PD_Hour",
        "EOT",
    ),

    (
        "Monthly Cumulative OT Hours",
        "MT_Hour",
        "OT",
    ),

    (
        "Monthly Cumultiave Extra OT Hours",
        "MT_Hour",
        "EOT",
    ),

    (
        "Monthly OT Expense",
        "MT_Amnt",
        "OT",
    ),

    (
        "Monthly Extra OT Expense",
        "MT_Amnt",
        "EOT",
    ),
]


REPORT_KPI_COLS = [
    "Number of OT Employees",
    "Number of Extra OT Employees",
    "Previous Days' OT Hours",
    "Previous Days' Extra OT Hours",
    "Monthly Cumulative OT Hours",
    "Monthly Cumultiave Extra OT Hours",
    "Monthly OT Expense",
    "Monthly Extra OT Expense",
]


# =============================================================================
# Source subtotal / total rows
# =============================================================================

_AGG_CATEGORY_KEYS = {
    "total",
    "factot",
    "facteot",
}

_AGG_FACTORY_KEYS = {
    "totalot",
    "totaleot",
    "grandtotal",
}


# =============================================================================
# Helper: normalize text for comparisons
# =============================================================================

def _norm_key(value):
    """
    Convert a value into a normalized alphabetic key.

    Examples
    --------
    'Fact OT'    -> 'factot'
    'Grand Total' -> 'grandtotal'
    'Woven'      -> 'woven'
    """
    return re.sub(
        r"[^a-z]",
        "",
        str(value).lower()
    )


# =============================================================================
# Helper: clean text values
# =============================================================================

def _clean_text(value):
    """
    Normalize whitespace and safely convert text values.
    """
    if pd.isna(value):
        return np.nan

    value = str(value)

    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value if value else np.nan


# =============================================================================
# Main report function
# =============================================================================

def build_ot_management_report(oteot):
    """
    Build OT / Extra-OT management report from cleaned `oteot`.

    Parameters
    ----------
    oteot : pandas.DataFrame
        Cleaned Category Wise OT report.

    Returns
    -------
    management_report : pandas.DataFrame
        Factory-wise management report containing exactly 8 KPI columns.

    report_info : dict
        Reporting period information.

    detail_kpi : pandas.DataFrame
        Detailed Factory + Type + Category KPI calculation.
    """

    # =========================================================================
    # 1. Validate input
    # =========================================================================

    if not isinstance(oteot, pd.DataFrame):
        raise TypeError(
            "`oteot` must be a pandas DataFrame."
        )

    required_rows = 3
    required_columns = max(COLS.values()) + 1

    if oteot.shape[0] < required_rows:
        raise ValueError(
            "`oteot` must contain at least 3 rows."
        )

    if oteot.shape[1] < required_columns:
        raise ValueError(
            f"`oteot` must contain at least "
            f"{required_columns} columns."
        )

    # =========================================================================
    # 2. Read reporting-period information from row 0
    # =========================================================================

    date_labels = []

    for value in oteot.iloc[0]:

        if isinstance(value, str):

            value = value.strip()

            if re.match(
                r"^\d{2}-[A-Za-z]{3}-\d{4}$",
                value
            ):
                date_labels.append(value)

    report_info = {}

    if date_labels:

        parsed_dates = pd.to_datetime(
            date_labels,
            format="%d-%b-%Y",
            errors="coerce"
        )

        parsed_dates = parsed_dates[
            ~parsed_dates.isna()
        ]

        if len(parsed_dates) > 0:

            parsed_dates = sorted(parsed_dates)

            report_info["Previous Day"] = (
                parsed_dates[-1].strftime("%d-%b-%Y")
            )

            report_info["Month Total Period"] = (
                f"{parsed_dates[0]:%d-%b-%Y} "
                f"to "
                f"{parsed_dates[-1]:%d-%b-%Y}"
            )

    # =========================================================================
    # 3. Extract actual data
    #
    # Row 0 -> reporting information
    # Row 1 -> headers
    # Row 2+ -> data
    # =========================================================================

    data = oteot.iloc[2:].copy()

    data = data.iloc[
        :,
        list(COLS.values())
    ].copy()

    data.columns = list(COLS.keys())

    # =========================================================================
    # 4. Clean text columns
    # =========================================================================

    TEXT_COLS = [
        "Factory",
        "Type",
        "Category",
        "OTEOT",
    ]

    for column in TEXT_COLS:

        data[column] = data[column].apply(
            _clean_text
        )

    # =========================================================================
    # 5. Remove completely invalid rows
    # =========================================================================

    data = data.dropna(
        subset=[
            "Factory",
            "Category",
            "OTEOT",
        ]
    ).reset_index(drop=True)

    # =========================================================================
    # 6. Convert numeric columns
    # =========================================================================

    for column in NUMERIC_COLS:

        data[column] = pd.to_numeric(
            data[column],
            errors="coerce"
        ).fillna(0)

    # =========================================================================
    # 7. Remove source subtotal / total rows
    # =========================================================================

    category_key = data["Category"].apply(
        _norm_key
    )

    factory_key = data["Factory"].apply(
        _norm_key
    )

    is_aggregate = (
        category_key.isin(_AGG_CATEGORY_KEYS)
        |
        factory_key.isin(_AGG_FACTORY_KEYS)
    )

    real = data.loc[
        ~is_aggregate
    ].reset_index(drop=True)

    if real.empty:

        raise ValueError(
            "No valid OT / Extra-OT category rows "
            "remained after removing subtotal and "
            "grand-total rows."
        )

    # =========================================================================
    # 8. Aggregate Factory + Type + Category + OT/EOT
    # =========================================================================

    grouped = (
        real
        .groupby(
            [
                "Factory",
                "Type",
                "Category",
                "OTEOT",
            ],
            sort=False,
            dropna=False,
        )[NUMERIC_COLS]
        .sum()
        .reset_index()
    )

    # =========================================================================
    # 9. Create OT / EOT pivot
    # =========================================================================

    pivot = grouped.pivot_table(
        index=[
            "Factory",
            "Type",
            "Category",
        ],

        columns="OTEOT",

        values=NUMERIC_COLS,

        fill_value=0,

        aggfunc="sum",
    )

    # =========================================================================
    # 10. Calculate detailed KPI
    # =========================================================================

    detail_kpi = pd.DataFrame(
        index=pivot.index
    )

    for kpi_name, source_column, ot_eot in KPI_SPEC:

        column_key = (
            source_column,
            ot_eot
        )

        if column_key in pivot.columns:

            detail_kpi[kpi_name] = (
                pivot[column_key]
            )

        else:

            detail_kpi[kpi_name] = 0

    detail_kpi = (
        detail_kpi
        .reset_index()
    )

    # =========================================================================
    # 11. Ensure numeric KPI columns
    # =========================================================================

    for column in REPORT_KPI_COLS:

        detail_kpi[column] = pd.to_numeric(
            detail_kpi[column],
            errors="coerce"
        ).fillna(0)

    # =========================================================================
    # 12. Factory + Type summary
    # =========================================================================

    type_summary = (
        detail_kpi
        .groupby(
            [
                "Factory",
                "Type",
            ],
            sort=False,
            dropna=False,
        )[REPORT_KPI_COLS]
        .sum()
        .reset_index()
    )

    # =========================================================================
    # 13. Build management report
    #
    # Logic:
    #
    # If a factory has Woven:
    #
    #     MGL
    #     MGL-Knit
    #
    # instead of:
    #
    #     MGL-Woven
    #     MGL-Knit
    #
    # If a factory has only one Type:
    #
    #     Factory
    #
    # is used directly.
    # =========================================================================

    rows = []

    BASE_TYPE_KEY = "woven"

    factory_order = list(
        dict.fromkeys(
            real["Factory"].tolist()
        )
    )

    for factory in factory_order:

        factory_types = (
            type_summary.loc[
                type_summary["Factory"] == factory,
                "Type"
            ]
            .tolist()
        )

        if not factory_types:
            continue

        # ---------------------------------------------------------------------
        # Find Woven type
        # ---------------------------------------------------------------------

        woven_type = next(
            (
                item
                for item in factory_types
                if _norm_key(item) == BASE_TYPE_KEY
            ),
            None
        )

        # ---------------------------------------------------------------------
        # Determine factory/base row
        # ---------------------------------------------------------------------

        if woven_type is not None:

            base_type = woven_type

        elif len(factory_types) == 1:

            base_type = factory_types[0]

        else:

            base_type = factory_types[0]

        # ---------------------------------------------------------------------
        # Get base type summary
        # ---------------------------------------------------------------------

        base_row = type_summary.loc[
            (
                type_summary["Factory"] == factory
            )
            &
            (
                type_summary["Type"] == base_type
            )
        ]

        if base_row.empty:
            continue

        base_values = (
            base_row.iloc[0]
            [REPORT_KPI_COLS]
            .to_dict()
        )

        rows.append(
            {
                "Factory": factory,
                **base_values,
            }
        )

        # ---------------------------------------------------------------------
        # Add other types
        # ---------------------------------------------------------------------

        for extra_type in factory_types:

            if extra_type == base_type:
                continue

            extra_row = type_summary.loc[
                (
                    type_summary["Factory"] == factory
                )
                &
                (
                    type_summary["Type"] == extra_type
                )
            ]

            if extra_row.empty:
                continue

            extra_values = (
                extra_row.iloc[0]
                [REPORT_KPI_COLS]
                .to_dict()
            )

            rows.append(
                {
                    "Factory": f"{factory}-{extra_type}",
                    **extra_values,
                }
            )

    # =========================================================================
    # 14. Grand Total
    # =========================================================================

    total_row = {
        "Factory": "Total"
    }

    for column in REPORT_KPI_COLS:

        total_row[column] = (
            detail_kpi[column].sum()
        )

    rows.append(total_row)

    # =========================================================================
    # 15. Create final management report
    # =========================================================================

    management_report = pd.DataFrame(
        rows,
        columns=[
            "Factory",
            *REPORT_KPI_COLS,
        ]
    )

    # =========================================================================
    # 16. Format numeric values
    # =========================================================================

    for column in REPORT_KPI_COLS:

        management_report[column] = pd.to_numeric(
            management_report[column],
            errors="coerce"
        )

    # =========================================================================
    # 17. Return only data — no Excel creation
    # =========================================================================

    return (
        management_report,
        report_info,
        detail_kpi,
    )


# =============================================================================
# Run
# =============================================================================
#
# `clean_category_wise_overtime()` should already be available from your
# cleaning cell.
# =============================================================================

oteot = clean_category_wise_overtime(
    input_file=r"Data\CategoryWiseOverTime.xlsx",
    source_sheet="Report_CategoryWiseOTWoven",
)


# =============================================================================
# Build report
# =============================================================================

management_report, report_info, detail_kpi = (
    build_ot_management_report(oteot)
)


# =============================================================================
# Final report
# =============================================================================

display(
    management_report
)

,Factory,Number of OT Employees,Number of Extra OT Employees,Previous Days' OT Hours,Previous Days' Extra OT Hours,Monthly Cumulative OT Hours,Monthly Cumultiave Extra OT Hours,Monthly OT Expense,Monthly Extra OT Expense
0,MGL,833.0,214.0,1658.5,467.0,31141.5,14481.5,2564833.26,1194652.95
1,MGL-Knit,264.0,20.0,518.5,64.0,17031.0,11937.0,1372749.99,964141.58
2,MGNSL,804.0,21.0,1595.0,61.0,32746.5,8840.5,2650018.29,716384.22
3,MGSL,466.0,1.0,911.5,1.0,25607.0,9025.0,2155418.90,758057.42
4,MGSLE,150.0,9.0,241.5,26.0,10810.0,4794.5,842328.07,373780.55
5,TMLS,29.0,24.0,55.5,30.5,1286.5,683.0,107508.64,58604.18
6,Total,2546.0,289.0,4980.5,649.5,118622.5,49761.5,9692857.15,4065620.90


In [2]:
# =============================================================================
# oteot -> OT / Extra-OT Management Report
# Returns pandas DataFrame only
# No Excel export / No file download
# =============================================================================

import re
import numpy as np
import pandas as pd


# =============================================================================
# Fixed zero-based column positions inside `oteot`
#
# Row 0 = Date / reporting-period information
# Row 1 = Actual column headers
# Row 2 onward = Data
# =============================================================================

COLS = {
    "Factory": 0,
    "Type": 1,
    "Category": 2,
    "OTEOT": 3,

    # Month Total
    "MT_Emp": 4,
    "MT_Hour": 5,
    "MT_Amnt": 6,

    # Previous Day
    "PD_Emp": 7,
    "PD_Hour": 8,
    "PD_Amnt": 9,
}


NUMERIC_COLS = [
    "MT_Emp",
    "MT_Hour",
    "MT_Amnt",
    "PD_Emp",
    "PD_Hour",
    "PD_Amnt",
]


# =============================================================================
# KPI definitions
# =============================================================================

KPI_SPEC = [
    (
        "Number of OT Employees",
        "PD_Emp",
        "OT",
    ),

    (
        "Number of Extra OT Employees",
        "PD_Emp",
        "EOT",
    ),

    (
        "Previous Days' OT Hours",
        "PD_Hour",
        "OT",
    ),

    (
        "Previous Days' Extra OT Hours",
        "PD_Hour",
        "EOT",
    ),

    (
        "Monthly Cumulative OT Hours",
        "MT_Hour",
        "OT",
    ),

    (
        "Monthly Cumultiave Extra OT Hours",
        "MT_Hour",
        "EOT",
    ),

    (
        "Monthly OT Expense",
        "MT_Amnt",
        "OT",
    ),

    (
        "Monthly Extra OT Expense",
        "MT_Amnt",
        "EOT",
    ),
]


REPORT_KPI_COLS = [
    "Number of OT Employees",
    "Number of Extra OT Employees",
    "Previous Days' OT Hours",
    "Previous Days' Extra OT Hours",
    "Monthly Cumulative OT Hours",
    "Monthly Cumultiave Extra OT Hours",
    "Monthly OT Expense",
    "Monthly Extra OT Expense",
]


# =============================================================================
# Source subtotal / total rows
# =============================================================================

_AGG_CATEGORY_KEYS = {
    "total",
    "factot",
    "facteot",
}

_AGG_FACTORY_KEYS = {
    "totalot",
    "totaleot",
    "grandtotal",
}


# =============================================================================
# Helper: normalize text for comparisons
# =============================================================================

def _norm_key(value):
    """
    Convert a value into a normalized alphabetic key.

    Examples
    --------
    'Fact OT'    -> 'factot'
    'Grand Total' -> 'grandtotal'
    'Woven'      -> 'woven'
    """
    return re.sub(
        r"[^a-z]",
        "",
        str(value).lower()
    )


# =============================================================================
# Helper: clean text values
# =============================================================================

def _clean_text(value):
    """
    Normalize whitespace and safely convert text values.
    """
    if pd.isna(value):
        return np.nan

    value = str(value)

    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value if value else np.nan


# =============================================================================
# Main report function
# =============================================================================

def build_ot_management_report(oteot):
    """
    Build OT / Extra-OT management report from cleaned `oteot`.

    Parameters
    ----------
    oteot : pandas.DataFrame
        Cleaned Category Wise OT report.

    Returns
    -------
    management_report : pandas.DataFrame
        Factory-wise management report containing exactly 8 KPI columns.

    report_info : dict
        Reporting period information.

    detail_kpi : pandas.DataFrame
        Detailed Factory + Type + Category KPI calculation.
    """

    # =========================================================================
    # 1. Validate input
    # =========================================================================

    if not isinstance(oteot, pd.DataFrame):
        raise TypeError(
            "`oteot` must be a pandas DataFrame."
        )

    required_rows = 3
    required_columns = max(COLS.values()) + 1

    if oteot.shape[0] < required_rows:
        raise ValueError(
            "`oteot` must contain at least 3 rows."
        )

    if oteot.shape[1] < required_columns:
        raise ValueError(
            f"`oteot` must contain at least "
            f"{required_columns} columns."
        )

    # =========================================================================
    # 2. Read reporting-period information from row 0
    # =========================================================================

    date_labels = []

    for value in oteot.iloc[0]:

        if isinstance(value, str):

            value = value.strip()

            if re.match(
                r"^\d{2}-[A-Za-z]{3}-\d{4}$",
                value
            ):
                date_labels.append(value)

    report_info = {}

    if date_labels:

        parsed_dates = pd.to_datetime(
            date_labels,
            format="%d-%b-%Y",
            errors="coerce"
        )

        parsed_dates = parsed_dates[
            ~parsed_dates.isna()
        ]

        if len(parsed_dates) > 0:

            parsed_dates = sorted(parsed_dates)

            report_info["Previous Day"] = (
                parsed_dates[-1].strftime("%d-%b-%Y")
            )

            report_info["Month Total Period"] = (
                f"{parsed_dates[0]:%d-%b-%Y} "
                f"to "
                f"{parsed_dates[-1]:%d-%b-%Y}"
            )

    # =========================================================================
    # 3. Extract actual data
    #
    # Row 0 -> reporting information
    # Row 1 -> headers
    # Row 2+ -> data
    # =========================================================================

    data = oteot.iloc[2:].copy()

    data = data.iloc[
        :,
        list(COLS.values())
    ].copy()

    data.columns = list(COLS.keys())

    # =========================================================================
    # 4. Clean text columns
    # =========================================================================

    TEXT_COLS = [
        "Factory",
        "Type",
        "Category",
        "OTEOT",
    ]

    for column in TEXT_COLS:

        data[column] = data[column].apply(
            _clean_text
        )

    # =========================================================================
    # 5. Remove completely invalid rows
    # =========================================================================

    data = data.dropna(
        subset=[
            "Factory",
            "Category",
            "OTEOT",
        ]
    ).reset_index(drop=True)

    # =========================================================================
    # 6. Convert numeric columns
    # =========================================================================

    for column in NUMERIC_COLS:

        data[column] = pd.to_numeric(
            data[column],
            errors="coerce"
        ).fillna(0)

    # =========================================================================
    # 7. Remove source subtotal / total rows
    # =========================================================================

    category_key = data["Category"].apply(
        _norm_key
    )

    factory_key = data["Factory"].apply(
        _norm_key
    )

    is_aggregate = (
        category_key.isin(_AGG_CATEGORY_KEYS)
        |
        factory_key.isin(_AGG_FACTORY_KEYS)
    )

    real = data.loc[
        ~is_aggregate
    ].reset_index(drop=True)

    if real.empty:

        raise ValueError(
            "No valid OT / Extra-OT category rows "
            "remained after removing subtotal and "
            "grand-total rows."
        )

    # =========================================================================
    # 8. Aggregate Factory + Type + Category + OT/EOT
    # =========================================================================

    grouped = (
        real
        .groupby(
            [
                "Factory",
                "Type",
                "Category",
                "OTEOT",
            ],
            sort=False,
            dropna=False,
        )[NUMERIC_COLS]
        .sum()
        .reset_index()
    )

    # =========================================================================
    # 9. Create OT / EOT pivot
    # =========================================================================

    pivot = grouped.pivot_table(
        index=[
            "Factory",
            "Type",
            "Category",
        ],

        columns="OTEOT",

        values=NUMERIC_COLS,

        fill_value=0,

        aggfunc="sum",
    )

    # =========================================================================
    # 10. Calculate detailed KPI
    # =========================================================================

    detail_kpi = pd.DataFrame(
        index=pivot.index
    )

    for kpi_name, source_column, ot_eot in KPI_SPEC:

        column_key = (
            source_column,
            ot_eot
        )

        if column_key in pivot.columns:

            detail_kpi[kpi_name] = (
                pivot[column_key]
            )

        else:

            detail_kpi[kpi_name] = 0

    detail_kpi = (
        detail_kpi
        .reset_index()
    )

    # =========================================================================
    # 11. Ensure numeric KPI columns
    # =========================================================================

    for column in REPORT_KPI_COLS:

        detail_kpi[column] = pd.to_numeric(
            detail_kpi[column],
            errors="coerce"
        ).fillna(0)

    # =========================================================================
    # 12. Factory + Type summary
    # =========================================================================

    type_summary = (
        detail_kpi
        .groupby(
            [
                "Factory",
                "Type",
            ],
            sort=False,
            dropna=False,
        )[REPORT_KPI_COLS]
        .sum()
        .reset_index()
    )

    # =========================================================================
    # 13. Build management report
    #
    # Logic:
    #
    # If a factory has Woven:
    #
    #     MGL
    #     MGL-Knit
    #
    # instead of:
    #
    #     MGL-Woven
    #     MGL-Knit
    #
    # If a factory has only one Type:
    #
    #     Factory
    #
    # is used directly.
    # =========================================================================

    rows = []

    BASE_TYPE_KEY = "woven"

    factory_order = list(
        dict.fromkeys(
            real["Factory"].tolist()
        )
    )

    for factory in factory_order:

        factory_types = (
            type_summary.loc[
                type_summary["Factory"] == factory,
                "Type"
            ]
            .tolist()
        )

        if not factory_types:
            continue

        # ---------------------------------------------------------------------
        # Find Woven type
        # ---------------------------------------------------------------------

        woven_type = next(
            (
                item
                for item in factory_types
                if _norm_key(item) == BASE_TYPE_KEY
            ),
            None
        )

        # ---------------------------------------------------------------------
        # Determine factory/base row
        # ---------------------------------------------------------------------

        if woven_type is not None:

            base_type = woven_type

        elif len(factory_types) == 1:

            base_type = factory_types[0]

        else:

            base_type = factory_types[0]

        # ---------------------------------------------------------------------
        # Get base type summary
        # ---------------------------------------------------------------------

        base_row = type_summary.loc[
            (
                type_summary["Factory"] == factory
            )
            &
            (
                type_summary["Type"] == base_type
            )
        ]

        if base_row.empty:
            continue

        base_values = (
            base_row.iloc[0]
            [REPORT_KPI_COLS]
            .to_dict()
        )

        rows.append(
            {
                "Factory": factory,
                **base_values,
            }
        )

        # ---------------------------------------------------------------------
        # Add other types
        # ---------------------------------------------------------------------

        for extra_type in factory_types:

            if extra_type == base_type:
                continue

            extra_row = type_summary.loc[
                (
                    type_summary["Factory"] == factory
                )
                &
                (
                    type_summary["Type"] == extra_type
                )
            ]

            if extra_row.empty:
                continue

            extra_values = (
                extra_row.iloc[0]
                [REPORT_KPI_COLS]
                .to_dict()
            )

            rows.append(
                {
                    "Factory": f"{factory}-{extra_type}",
                    **extra_values,
                }
            )

    # =========================================================================
    # 14. Grand Total
    # =========================================================================

    total_row = {
        "Factory": "Total"
    }

    for column in REPORT_KPI_COLS:

        total_row[column] = (
            detail_kpi[column].sum()
        )

    rows.append(total_row)

    # =========================================================================
    # 15. Create final management report
    # =========================================================================

    management_report = pd.DataFrame(
        rows,
        columns=[
            "Factory",
            *REPORT_KPI_COLS,
        ]
    )

    # =========================================================================
    # 16. Format numeric values
    # =========================================================================

    for column in REPORT_KPI_COLS:

        management_report[column] = pd.to_numeric(
            management_report[column],
            errors="coerce"
        )

    # =========================================================================
    # 17. Return only data — no Excel creation
    # =========================================================================

    return (
        management_report,
        report_info,
        detail_kpi,
    )


# =============================================================================
# Run
# =============================================================================
#
# `clean_category_wise_overtime()` should already be available from your
# cleaning cell.
# =============================================================================

oteot = clean_category_wise_overtime(
    input_file=r"..\Data\CategoryWiseOverTime.xlsx",
    source_sheet="Report_CategoryWiseOTWoven",
)


# =============================================================================
# Build report
# =============================================================================

oteot_report, report_info, detail_kpi = (
    build_ot_management_report(oteot)
)


# =============================================================================
# Final report
# =============================================================================

display(
    oteot_report
)

NameError: name 'clean_category_wise_overtime' is not defined